# ML-05 — Feature Vector and Leakage/Privacy Check

This notebook builds our ML feature vector from the warehouse database, defines the target label (persistent organic search volume decline), performs an honest validation split check, and executes leakage tests to secure the modeling pipeline.

## 1. Build the feature vector

We connect to DuckDB, authenticate to the Hugging Face dataset, and perform aggregations inside DuckDB. We define features strictly in the historical feature window, and evaluate the target label in the future outcome window relative to the panel's maximum date.

In [1]:
import duckdb
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import classification_report

# Retrieve token from the environment variable
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN environment variable is not set. Please set it to proceed.")

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute("SET enable_server_cert_verification = false;")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':      f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':      f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':       f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':   f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("DuckDB registered and authenticated.")

DuckDB registered and authenticated.


In [2]:
# Query and build aggregated feature vectors at the page grain
# Feature window is report_date <= end_d - 30 (Historical)
# Outcome window is report_date > end_d - 30 (Future) - Used for Label
# Having threshold: at least 100 historical impressions to stabilize trends.
raw_features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    daily_agg AS (
        SELECT 
            f.client_hash_id, 
            f.content_hash_id,
            -- Historical Feature Window metrics (report_date <= end_d - 30)
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END) AS clk_prev30,
            AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END) AS pos_prev30,
            
            -- Future Outcome Window metrics (report_date > end_d - 30) - USED FOR LABEL
            SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
            SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END) AS clk_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM daily_agg
""").df()

# Query-Mix features from fact_query_90d
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

# Derived query-mix metrics
qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
qsignals = qsignals.drop(columns=['top_query_impressions', 'kept_impressions'])

# Content characteristics from dim_content
content_meta = con.sql(f"""
    SELECT content_hash_id, word_count, content_type
    FROM {TABLES['dim_content']}
""").df()

# Merge all datasets
data = raw_features.merge(qsignals, on='content_hash_id', how='left')
data = data.merge(content_meta, on='content_hash_id', how='left')

# Handle missing values using missingness indicator flags to avoid proxy leakage
data['word_count_missing'] = data['word_count'].isna().astype(int)
data['word_count'] = data['word_count'].fillna(0)
data['anon_share_missing'] = data['anon_share'].isna().astype(int)
data['anon_share'] = data['anon_share'].fillna(0)
data['rare_share_missing'] = data['rare_share'].isna().astype(int)
data['rare_share'] = data['rare_share'].fillna(0)
data['visible_queries'] = data['visible_queries'].fillna(0)
data['top_query_share'] = data['top_query_share'].fillna(0)

# Convert categoricals safely
data = pd.get_dummies(data, columns=['content_type'], drop_first=True)

# Define the target label: true traffic decline (> 20% drop in impressions)
# Formula: imp_last30 < 0.8 * imp_prev30
data['is_declining_label'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

print(f"Feature vector created successfully: {data.shape[0]:,} records, {data.shape[1]} features.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector created successfully: 111,247 records, 18 features.


## 2. Feature notes (meaning, missing, categorical, available-when?)

We list and define the model features below, verifying their availability before the decision moment $t$ (end_d - 30 days).

In [3]:
# List features and confirm availability
feature_cols = [
    'imp_prev30', 'clk_prev30', 'pos_prev30', 
    'visible_queries', 'rare_share', 'anon_share', 'top_query_share',
    'word_count', 'word_count_missing', 'anon_share_missing', 'rare_share_missing'
]
# Add dummy variables for content_type
content_type_cols = [c for c in data.columns if c.startswith('content_type_')]
feature_cols.extend(content_type_cols)

print("Features available strictly BEFORE prediction window:")
for f in feature_cols:
    print(f" - {f}")

Features available strictly BEFORE prediction window:
 - imp_prev30
 - clk_prev30
 - pos_prev30
 - visible_queries
 - rare_share
 - anon_share
 - top_query_share
 - word_count
 - word_count_missing
 - anon_share_missing
 - rare_share_missing
 - content_type_feedly article
 - content_type_keyword article


## 3. The leakage hunt

We audit our model for leakage. First, we perform an honest evaluation using GroupShuffleSplit on client_hash_id to check how the model generalizes to unseen clients. Next, we execute an intentional leakage test by adding a sibling metric (future clicks `clk_last30`) to prove our validation pipeline catches leakage.

In [4]:
# Target variable
y = data['is_declining_label']
groups = data['client_hash_id']

# A. Honest Grouped Split
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(data, y, groups=groups))

X_clean = data[feature_cols]
X_train_g, X_test_g = X_clean.iloc[train_idx], X_clean.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

model_clean = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model_clean.fit(X_train_g, y_train_g)
y_pred_g = model_clean.predict(X_test_g)

print("=== HONEST GROUPED SPLIT (BY CLIENT) ===")
base_rate_test = max(y_test_g.mean(), 1 - y_test_g.mean())
print(f"Base Rate (majority class): {base_rate_test:.3f}")
print(classification_report(y_test_g, y_pred_g, digits=3))

# B. Standard Random Split (showing the memorization gap)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_clean, y, test_size=0.3, random_state=42, stratify=y
)
model_random = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model_random.fit(X_train_r, y_train_r)
y_pred_r = model_random.predict(X_test_r)

print("=== STANDARD RANDOM SPLIT ===")
print(classification_report(y_test_r, y_pred_r, digits=3))

=== HONEST GROUPED SPLIT (BY CLIENT) ===
Base Rate (majority class): 0.671
              precision    recall  f1-score   support

           0      0.468     0.433     0.450      9927
           1      0.732     0.758     0.745     20259

    accuracy                          0.651     30186
   macro avg      0.600     0.596     0.597     30186
weighted avg      0.645     0.651     0.648     30186



=== STANDARD RANDOM SPLIT ===
              precision    recall  f1-score   support

           0      0.625     0.444     0.519     11910
           1      0.734     0.852     0.789     21465

    accuracy                          0.707     33375
   macro avg      0.680     0.648     0.654     33375
weighted avg      0.695     0.707     0.693     33375



In [5]:
# C. Intentional Leakage Test (introducing clk_last30 which is in the future label window)
leaky_feature_cols = feature_cols + ['clk_last30']
X_leaky = data[leaky_feature_cols]

X_train_l, X_test_l = X_leaky.iloc[train_idx], X_leaky.iloc[test_idx]

model_leaky = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model_leaky.fit(X_train_l, y_train_g)
y_pred_l = model_leaky.predict(X_test_l)

print("=== INTENTIONAL LEAKAGE SPLIT (SIBLING METRIC LEAK) ===")
print(classification_report(y_test_g, y_pred_l, digits=3))

# D. Compare Feature Importances to prove leak towers over others
importances = pd.Series(model_leaky.feature_importances_, index=leaky_feature_cols)
print("\nTop 5 Feature Importances in Leaky Model:")
print(importances.sort_values(ascending=False).head(5))

=== INTENTIONAL LEAKAGE SPLIT (SIBLING METRIC LEAK) ===
              precision    recall  f1-score   support

           0      0.685     0.666     0.675      9927
           1      0.838     0.850     0.844     20259

    accuracy                          0.790     30186
   macro avg      0.762     0.758     0.760     30186
weighted avg      0.788     0.790     0.789     30186


Top 5 Feature Importances in Leaky Model:
clk_last30    0.163438
imp_prev30    0.154427
pos_prev30    0.127039
rare_share    0.109753
anon_share    0.107893
dtype: float64


## 4. What I excluded and why

We refuse to use the following fields as model features:

1. `client_hash_id`: Excluded because it is our grouping entity; using it as an input feature would encourage the model to memorize client-specific biases instead of generalizable traffic dynamics.
2. `content_hash_id` / `content_id`: Excluded due to extremely high cardinality. Feeding this as a feature results in client/page memorization and high generalization loss.
3. `imp_last30` / `clk_last30`: Excluded because they are sibling metrics inside the future label window $[t+1, t+30]$, introducing direct target leakage.
4. `provider_used` / `model_used`: Excluded because the generative model choice is metadata, not an organic indicator of ranking decay.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.